# Project diary: England housing affordability pipeline

My notes as I build this project: what I did, what I found out, what I learnt, and what I'd do differently. The aim is to understand data analysis properly, not just get the code working.

**The question:** how affordable is housing across England's councils, and how has that changed since 2018?

**The data:** every residential sale in England from HM Land Registry (about 7 million rows for 2018–2024), and the ONS tables comparing house prices with earnings.

## Step 0: getting set up

**What I did:** installed Python 3.12, PostgreSQL 17 (with pgAdmin) and Power BI Desktop, and checked that Git, the GitHub CLI and VS Code were already there.

**What I learnt**
- A project depends on a whole toolchain, not just code. Someone else should be able to rebuild mine from a list, which is why `requirements.txt` and a README matter.
- PostgreSQL is a server that runs in the background. Python, psql and pgAdmin are all just different ways of talking to it.

**Reflection:** I underestimated setup. In a real team, "it works on my machine" is a genuine problem, and the environment is part of the project.

## Step 1: project skeleton, virtual environment and database

**What I did**
- Set up the folder structure, `.gitignore`, a virtual environment, the `housing` database with its own user, and a `.env` file for the connection settings.
- Wrote `db.py` to connect to the database, and a small command line (`python -m pipeline run --years ...`).
- Opened and merged my first pull request.

**What went wrong**
- My scripts printed nothing because I'd never saved the files. They were empty on disk.
- PowerShell blocked the venv's activate script until I allowed local scripts to run.
- `__init.py` needed two underscores on each side.

**What I learnt**
- **Virtual environments** keep each project's libraries separate. I commit the recipe (`requirements.txt`), not the libraries themselves.
- **Secrets never go in code.** The password lives in `.env`, which Git ignores. `.env.example` shows what's needed without the real values.
- **`.gitignore` first.** It's much easier to keep data and secrets out of Git than to remove them later.

**Reflection:** most of my early "bugs" weren't code problems at all. I'm learning to check the boring things first: is it saved, am I in the right folder, is the venv active?

## Step 2: getting the raw data in

**What I did**
- Created one schema per layer: `raw`, `staging`, `core`, `mart`, plus `etl` for logging.
- Wrote a downloader that skips files it already has and retries if the connection drops.
- Loaded the Land Registry CSVs and the ONS spreadsheet into `raw`, keeping every column as text.
- Logged every step (rows in, rows out) to a run log.
- Decided to include **2018** as well, to get a pre-pandemic baseline.

**What I found out**
- 7,100,023 sales were loaded, ranging from about 860k in 2023 to about 1.28 million in 2021. That's already a hint that 2021 was unusual.
- The ONS spreadsheet is built for people, not computers: title rows, notes, a column per year, and markers like `[x]` instead of numbers.

**What went wrong**
- I typed `raw_price_paid` instead of `raw.price_paid`, so the table ended up in the wrong schema.
- I ran the SQL files in the wrong order, so the schema didn't exist yet.
- I forgot to download the ONS file before loading it.

**What I learnt**
- **Keep raw data untouched.** Loading everything as text means a bad value can't stop the load. If I make a mistake later, I can rebuild from raw without downloading again.
- **Reshaping wide to long.** One row per council per year is much easier to join, filter and chart than a column per year.
- **Lineage.** Tagging each row with the file it came from means any number can be traced back to its source.
- **Parameterised queries** (`%s`) keep data separate from SQL. That's how SQL injection is prevented, and it also stops an apostrophe in a name breaking a query.

**Reflection:** most of the work in data analysis is getting data into a usable shape before any analysis happens. I also realised the real question isn't "did it load?" but "did all of it load?" Counting rows at every step is what makes me trust the result.

## Step 3: cleaning and matching

**What I did**
- Cleaned the ONS tables into proper numbers and years.
- Matched each Land Registry district to an ONS council, automatically where the names agree, otherwise from a seed file.
- Wrote tests for the name matching.
- Split every sale into **kept** or **rejected**, with a reason, and checked that the two always add back up to the raw total.

**What I found out**
- **46 districts didn't match**, because councils have merged and been renamed. For example, Harrogate and Scarborough became North Yorkshire (2023), and Aylesbury Vale became Buckinghamshire (2020). The same place can have different names in different datasets and different years.
- **1,158,957 sales were "category B"** (repossessions, buy-to-let and other non-standard sales), about 16% of everything. I rejected them so my figures follow the same rules as ONS.
- **307,577 sales were in Wales.** They were rejected because this is an England project.
- **No rows had a bad price or date.** The government data was cleaner than I expected.
- **861 sales were extreme** (under £10k or over £10m). I flagged them rather than deleting them.

**What I learnt**
- **Joining datasets is where things quietly go wrong.** If I'd just done a normal join, 46 districts' worth of sales would have silently disappeared. Making the pipeline stop when anything is unmatched turns a silent error into a loud one.
- **Reject, don't delete.** Keeping rejected rows with a reason means I can always explain where every row went.
- **Flag, don't remove.** Outliers might be real (a £15m mansion is a real sale). Flagging lets me decide per analysis whether to include them.
- **Tests** check my assumptions automatically, so I don't have to remember them.

**Reflection:** cleaning isn't one step. It's lots of small decisions, and each one changes the answer. Writing down *why* I made each decision (the seed file notes, the reject reasons) is what makes the analysis defensible.

## Step 4: building the model

**What I did**
- Built a **star schema**: a fact table of sales, surrounded by dimension tables for date, council and property type.
- Built **marts**, ready-made summary tables by council-year and council-month, for Power BI.
- Added checks that rerun the reconciliation at any time, and a `check` command.

**What I found out**
- 5,633,489 sales ended up in the fact table. That's exactly the staged count, so nothing was lost or duplicated in the joins.
- The marts cover 296 English councils for each year from 2019 to 2024.

**What I learnt**
- **Fact vs dimension.** Facts are the things you count or sum (sales, prices). Dimensions are the things you slice them by (when, where, what type). Most business questions are "a fact, sliced by dimensions".
- **ONS years run October to September**, not January to December. If I hadn't matched their definition, my numbers would never have lined up with theirs.
- **Building and checking are separate jobs.** I can re-check the database without rebuilding it.

**Reflection:** I now see why the layers exist. Raw is for trust, staging is for cleaning, core is for modelling, and marts are for answering questions. Each layer has one job, which makes problems easier to find.

## Step 5: profiling, looking before concluding

**What I did:** before any analysis, I checked whether the data looks believable. For each check I wrote down what I expected, drew it, and then compared.

**What I found out**
- **The data reflects real events.** A normal month in 2019 had about 66,000 sales. In April 2020 (lockdown) it fell to about 24,000. Sales then peaked at about 160,000 in June 2021 and 126,000 in March 2021, exactly the two stamp duty holiday deadlines.
- **Prices are very skewed.** The 2023 median was £292,500, but a few very expensive sales stretch the scale. A log scale shows the shape properly.
- **The property mix shifted.** In 2020 detached houses rose to 28% of sales and flats fell to 17% (the "race for space"). So some of the 2020–21 price rise is *which* homes sold, not just homes getting more expensive.
- **Tiny councils are unreliable.** The Isles of Scilly had as few as 10 sales in a year, and a median from 10 sales means very little.
- **My numbers match the official ones.** 99.7% of council-years are within 2% of the ONS median price.

**What I learnt**
- **Write the expectation down first.** It stops me explaining away whatever I see after the fact.
- **The median beats the average** for prices, because a few very expensive homes drag the average up.
- **Composition effect.** A median can change even when no individual price does, if the mix of what's sold changes. I need to be careful saying "prices rose".
- **Sample size matters.** Small groups produce noisy numbers, so they should be left out of rankings.
- **Validating against a trusted source** (ONS) is the strongest evidence that my pipeline is right.

**Limitations so far**
- 2020 and 2021 are distorted by lockdown and the stamp duty holiday.
- Category B sales are excluded, so this covers standard market sales only.
- 2018 only partly counts, because ONS years start in October.

**Reflection:** profiling changed how I see analysis. The interesting findings (the lockdown dip, the stamp duty spikes, the mix shift) came from *checking* the data, not from the final question. A good analyst spends time understanding the data before trusting it.

## Data quality summary

| Measure | Value |
|---|---|
| Sales loaded from Land Registry | 7,100,023 |
| Rejected: category B | 1,158,957 |
| Rejected: Wales (out of scope) | 307,577 |
| Rejected: bad price or date | 0 |
| Kept but flagged as outliers | 861 |
| Districts matched by hand | 46 |
| Council-years within 2% of ONS | 99.7% |